# 03 — Final evaluation and ONNX export

Run this only after selecting the checkpoint on validation CER. It evaluates the untouched Assamese test split, exports dynamic-width ONNX, quantizes weights, verifies numerical parity, and measures CPU latency.

## Assamese Tesseract baseline

Install the Tesseract executable on the notebook host, then measure the official `asm` model on exactly the same test split.

In [ ]:
import shutil
import subprocess

if shutil.which("tesseract") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr"], check=True)
!python scripts/download_tesseract_baselines.py
!python scripts/evaluate_tesseract.py --profile fast --split test

In [ ]:
!python scripts/evaluate_recognizer.py --checkpoint artifacts/recognizer/best.pt --dataset data/processed/mozhi_assamese --split test

In [ ]:
!python scripts/export_recognizer.py --checkpoint artifacts/recognizer/best.pt --output artifacts/recognizer/assamese_recognizer.onnx --opset 18 --quantize

In [ ]:
!python scripts/benchmark_onnx.py --model artifacts/recognizer/assamese_recognizer.onnx --width 384
!python scripts/benchmark_onnx.py --model artifacts/recognizer/assamese_recognizer.int8.onnx --width 384

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("artifacts/recognizer/test_metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics["metrics"], ensure_ascii=False, indent=2))
print("Copy these measured values into docs/MODEL_CARD_TEMPLATE.md.")